# Ensemble methods on Titanic 🚢🚢

## Introduction

This exercise is the opportunity to practice ensemble methods on a dataset you have worked with before and that is the Titanic dataset.

Let's start by importing the librairies that we will used in the exercise.

In [37]:
#!pip install -q xgboost
#!pip install -q s3fs

In [38]:
# Load in our libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "svg" # to be replaced by "iframe" if working on JULIE

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
# import ensemble methods
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

## Feature Exploration, Engineering and Cleaning

1. Import the data using the following link : "https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+Supervis%C3%A9/stacking/titanic.csv" , and display the first lines. Are there any missing values in the dataset?

In [39]:
import pandas as pd

url = "https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+Supervis%C3%A9/stacking/titanic.csv"

df = pd.read_csv(url)

df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [40]:
df.isna().sum()


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [41]:
df.isna().sum().sum()

np.int64(866)

2. What types of variables are present in this dataset? What kind of preprocessing could you run on these variables?

In [42]:
# Step — Detect names of numeric and categorical features

print(df.select_dtypes(include=["int", "float"]).columns.tolist()) # toujours int et float sinon pas de month etc...

['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']


In [43]:
print(df.select_dtypes(include=["boolean"]).columns.tolist())

[]


In [44]:
print(df.select_dtypes(include=["object"]).columns.tolist())

['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


In [45]:
# Identify categorical columns with too many categories (useless for small datasets)
categorical_cols = df.select_dtypes(include='object').columns

print("\n--- Unique categories count ---")
for col in categorical_cols:
    print(f"{col} : {df[col].nunique()} unique values")



--- Unique categories count ---
Name : 891 unique values
Sex : 2 unique values
Ticket : 681 unique values
Cabin : 147 unique values
Embarked : 3 unique values


In [46]:
numeric_features = df.select_dtypes(include=['float', 'int']).columns.drop("Survived")
categorical_features = df.select_dtypes(include='object').columns

In [47]:
numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer()),
    # le knn remplace une valeur manquante par les k obeservations les plus proches selon une distance euclidienne
    # tandis que le simple imputer ne remplace les valerus manquantes que par une valeur unique
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


3. Here are some guidelines you can follow to clean the dataset as well as create new variables (feature engineering).

a.  Create a Name_length variable that measures the number of characters in the variable Name for each observations.

In [48]:
df["Name"].head()

,Name
0,"Braund, Mr. Owen Harris"
1,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,"Heikkinen, Miss. Laina"
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"
4,"Allen, Mr. William Henry"


In [49]:
df["Name_length"] = df["Name"].str.len()
# Rappel
# df["name"] renvoie une série or 'Series' object has no attribute 'len'
# str permet d'appliqer des opérations de string à chaque élément
df["Name_length"].head()

,Name_length
0,23
1,51
2,22
3,44
4,24


b. Create a variable Has_Cabin that indicates whether the passenger has a cabin or not.

Hint: [this method](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.notna.html#pandas.DataFrame.notna) might be useful 😉

In [50]:
df["Has_Cabin"] = df["Cabin"].notna()
df["Has_Cabin"].head()

,Has_Cabin
0,False
1,True
2,False
3,True
4,False


c. Create a variable FamilySize that gives the size of each passenger's family.

In [51]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["FamilySize"].head()
# + 1 pour le passager lui-meme

,FamilySize
0,2
1,2
2,1
3,2
4,1


d. Create a variable IsAlone that indicates whether the passenger is traveling on their own.

In [52]:
df["IsAlone"] = df["FamilySize"] == 1
df["IsAlone"].head()

,IsAlone
0,False
1,False
2,True
3,False
4,True


h. Extract the title from each passenger in order to create a variable Title.

Hint: You might consider _applying_ a function that calls the [str.split method](https://docs.python.org/3.3/library/stdtypes.html?highlight=split#str.split) 😉

In [53]:
df["Name"].head()

,Name
0,"Braund, Mr. Owen Harris"
1,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,"Heikkinen, Miss. Laina"
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"
4,"Allen, Mr. William Henry"


In [54]:
df["Title"] = df["Name"].str.split(",").str[1].str.split(".").str[0].str.strip()
df["Title"].head()

,Title
0,Mr
1,Mrs
2,Miss
3,Mrs
4,Mr


In [55]:
# df["Title"] = (
#     df["Name"]
#     .str.extract(r",\s*([^.]*)\.", expand=False)   # Regex robuste
#     .str.strip()
# )


i. If some of these titles are equivalent convert them in order to bring them all in the same category.

In [56]:
df["Title"].nunique()
# df["Title"].unique()
# len(df["Title"].value_counts())

17

In [57]:
df["Title"].value_counts()



,count
Title,
Mr,517
Miss,182
Mrs,125
Master,40
Dr,7
Rev,6
Col,2
Mlle,2
Major,2


j. Are any of the remaining titles underrepresented among the observations? If it is the case, group them in a unique modality "Rare"

In [58]:
title_map = {
    # Normalisation des variantes linguistiques
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",

    # Titres rares regroupés
    "Dr": "Rare",
    "Rev": "Rare",
    "Col": "Rare",
    "Major": "Rare",
    "Don": "Rare",
    "Lady": "Rare",
    "Sir": "Rare",
    "Capt": "Rare",
    "the Countess": "Rare",
    "Jonkheer": "Rare"
}

df["Title"] = df["Title"].replace(title_map)
df["Title"].value_counts()

,count
Title,
Mr,517
Miss,185
Mrs,126
Master,40
Rare,23


4. Drop the columns 'PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp' du dataset. Why don't we need these columns for what's next?

In [59]:
columns_to_drop = ["PassengerId", "Name", "Ticket", "Cabin", "SibSp"]

df_clean = df.drop(columns=columns_to_drop)

print("\n--- Remaining columns after dropping useless ones ---")
print(df_clean.head())


--- Remaining columns after dropping useless ones ---
   Survived  Pclass     Sex   Age  Parch     Fare Embarked  Name_length  \
0         0       3    male  22.0      0   7.2500        S           23   
1         1       1  female  38.0      0  71.2833        C           51   
2         1       3  female  26.0      0   7.9250        S           22   
3         1       1  female  35.0      0  53.1000        S           44   
4         0       3    male  35.0      0   8.0500        S           24   

   Has_Cabin  FamilySize  IsAlone Title  
0      False           2    False    Mr  
1       True           2    False   Mrs  
2      False           1     True  Miss  
3       True           2    False   Mrs  
4      False           1     True    Mr  


5. Separate the features from the target and split the data between train and test (with random_state = 0)

In [60]:
X = df_clean.drop(columns=["Survived"])
y = df_clean["Survived"]
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)


6. Using the Pipeline and ColumnTransformer, make all the preprocessings at once. Use the [KNN imputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html) to handle the missing values in the numeric variables, and the SimpleImputer for categorical data.

In [61]:
numeric_features = [
    "Pclass",
    "Age",
    "Parch",
    "Fare",
    "Name_length",
    "FamilySize"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Has_Cabin",
    "IsAlone",
    "Title"
]
numeric_transformer = Pipeline(steps=[
    ("imputer", KNNImputer()),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [62]:
# Fit on train, transform train
print("Performing preprocessings on train set...")
X_train_preprocessed = preprocessor.fit_transform(X_train)
print("...Done.")

# Transform test set only (no fit!)
print("\nPerforming preprocessings on test set...")
X_test_preprocessed = preprocessor.transform(X_test)
print("...Done.")


Performing preprocessings on train set...
...Done.

Performing preprocessings on test set...
...Done.


### Pearson Correlation Heatmap

7. Produce a figure that contains the correlation table for all the explanatory variables of X_train, what do you think?

Rappel
La correlation ne s'applique qu'aux colonnes numériques


In [63]:
 #!pip install -U kaleido

In [64]:
corr_matrix = X_train[numeric_features].corr()
import plotly.io as pio
pio.renderers.default = "colab" # "notebook " "# pour que colab ne bloque pas l'export svg

fig = px.imshow(
    corr_matrix,
    text_auto=".3f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    title="Pearson Correlation Heatmap - X_train"
)

fig.update_layout(width=900, height=700)
fig.show() # fig.show(rendereres ...)


**Correlations between the variables are not very high, we can hope that they will each bring complementary information in order to predict the target variable.**

## Ensembling & Stacking models

Now that we have finished our preprocessing and made sure our data was fit for prediction, let's move on to creating our ensemble models. We'll train different models with different ensembling strategies and store their train and test scores for comparison.

### Random Forest
8. Train a Random Forest by tuning the hyperparameters with a grid search. Which ensemble method is related to random forests?

Evaluate the best model's accuracy on train and test sets. Save the scores into a pandas DataFrame.

Méthodes d'ensemble =>


Random Forest => famille des méthodes de type Bagging (Bootstrap Aggregation).
qui :

- génère de nombreux arbres entraînés sur des échantillons bootstrap

- combine leurs prédictions par vote majoritaire

- ajoute une randomisation supplémentaire sur les features → Random Subspaces

En substance
Random Forest = Bagging + Sélection aléatoire de variables.

In [65]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(random_state=0))
])


In [66]:
rf_params = {
    "model__n_estimators": [10, 20, 40, 60, 80, 100],
    "model__max_depth": [2, 4, 6, 8, 10],
    "model__min_samples_split": [2, 4, 8],
    "model__min_samples_leaf": [1, 2, 5]
}
print("Grid search...")

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("...Done.")
print("Best hyperparameters :", rf_grid.best_params_)
print("Best validation accuracy :", rf_grid.best_score_)
# Best model
best_rf = rf_grid.best_estimator_

train_acc = best_rf.score(X_train, y_train)
test_acc = best_rf.score(X_test, y_test)

print("\nAccuracy on training set :", train_acc)
print("Accuracy on test set :", test_acc)

results = pd.DataFrame({
    "model": ["random_forest", "random_forest"],
    "accuracy": [train_acc, test_acc],
    "set": ["train", "test"]
})

results



Grid search...
...Done.
Best hyperparameters : {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 10}
Best validation accuracy : 0.828635251568982

Accuracy on training set : 0.9466292134831461
Accuracy on test set : 0.7988826815642458


,model,accuracy,set
0,random_forest,0.946629,train
1,random_forest,0.798883,test


9. Create your own Bagging of decision tree (with the same hyperparameters as the optimal ones for Random Forest) and check you get compatible performances.

Bagging => plusieurs arbres indépendants

In [67]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# Decision Tree with the same hyperparameters as the optimal Random Forest
dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=0
)

# Bagging classifier using the above decision tree
bagging = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", BaggingClassifier(
        estimator=dt_model,
        n_estimators=10,       # same number of trees as optimal RF
        random_state=0,
        n_jobs=-1
    ))
])

print("Training Bagging of decision tree...")
bagging.fit(X_train, y_train)
print("...Done.")

# Accuracy on train and test
bag_train_acc = bagging.score(X_train, y_train)
bag_test_acc = bagging.score(X_test, y_test)

print("Accuracy on training set :", bag_train_acc)
print("Accuracy on test set :", bag_test_acc)
results = pd.concat([
    results,  # contains RF results
    pd.DataFrame({
        "model": ["bagging_dt", "bagging_dt"],
        "accuracy": [bag_train_acc, bag_test_acc],
        "set": ["train", "test"]
    })
], ignore_index=True)

results



Training Bagging of decision tree...
...Done.
Accuracy on training set : 0.9564606741573034
Accuracy on test set : 0.8268156424581006


,model,accuracy,set
0,random_forest,0.946629,train
1,random_forest,0.798883,test
2,bagging_dt,0.956461,train
3,bagging_dt,0.826816,test


10. Train scikit-learn's GradientBoosting model (by tuning hyperparameters) and evaluate the performances.

Gradient Boosting =>



In [68]:
from sklearn.ensemble import GradientBoostingClassifier

gb_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", GradientBoostingClassifier(random_state=0))
])
gb_params = {
    "model__n_estimators": [2, 4, 6, 8, 10, 12],
    "model__max_depth": [8, 10, 12],
    "model__min_samples_split": [6, 8, 10],
    "model__min_samples_leaf": [1, 2, 3]
}
print("Grid search...")

gb_grid = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

gb_grid.fit(X_train, y_train)

print("...Done.")
print("Best hyperparameters :", gb_grid.best_params_)
print("Best validation accuracy :", gb_grid.best_score_)

best_gb = gb_grid.best_estimator_

gb_train_acc = best_gb.score(X_train, y_train)
gb_test_acc = best_gb.score(X_test, y_test)

print("\nAccuracy on training set :", gb_train_acc)
print("Accuracy on test set :", gb_test_acc)
results = pd.concat([
    results,
    pd.DataFrame({
        "model": ["gradient_boost", "gradient_boost"],
        "accuracy": [gb_train_acc, gb_test_acc],
        "set": ["train", "test"]
    })
], ignore_index=True)

results


Grid search...
...Done.
Best hyperparameters : {'model__max_depth': 12, 'model__min_samples_leaf': 2, 'model__min_samples_split': 6, 'model__n_estimators': 4}
Best validation accuracy : 0.8230152820621921

Accuracy on training set : 0.922752808988764
Accuracy on test set : 0.7988826815642458


,model,accuracy,set
0,random_forest,0.946629,train
1,random_forest,0.798883,test
2,bagging_dt,0.956461,train
3,bagging_dt,0.826816,test
4,gradient_boost,0.922753,train
5,gradient_boost,0.798883,test


11. Train an XGBoost model (by tuning hyperparameters). Do you get better or similar results compared to scikit-learn's GradientBoosting?

In [69]:
from xgboost import XGBClassifier
xgb_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(
        objective="binary:logistic",
        # xgboost doit savoir quelle fonction de perte utiliser
        # s'il s'agit d'une classification
        eval_metric="logloss",
        random_state=0,
        use_label_encoder=False
    ))
])

xgb_params = {
    "model__n_estimators": [2, 4, 6, 8, 10, 12],
    "model__max_depth": [4, 6, 8, 10],
    "model__min_child_weight": [1, 2, 4, 6, 8]
}
print("Grid search...")

xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

xgb_grid.fit(X_train, y_train)

print("...Done.")
print("Best hyperparameters :", xgb_grid.best_params_)
print("Best validation accuracy :", xgb_grid.best_score_)
best_xgb = xgb_grid.best_estimator_

xgb_train_acc = best_xgb.score(X_train, y_train)
xgb_test_acc = best_xgb.score(X_test, y_test)

print("\nAccuracy on training set :", xgb_train_acc)
print("Accuracy on test set :", xgb_test_acc)

results = pd.concat([
    results,
    pd.DataFrame({
        "model": ["xgboost", "xgboost"],
        "accuracy": [xgb_train_acc, xgb_test_acc],
        "set": ["train", "test"]
    })
], ignore_index=True)

results


Grid search...
...Done.
Best hyperparameters : {'model__max_depth': 6, 'model__min_child_weight': 2, 'model__n_estimators': 8}
Best validation accuracy : 0.8300476308666926

Accuracy on training set : 0.8946629213483146
Accuracy on test set : 0.8100558659217877


,model,accuracy,set
0,random_forest,0.946629,train
1,random_forest,0.798883,test
2,bagging_dt,0.956461,train
3,bagging_dt,0.826816,test
4,gradient_boost,0.922753,train
5,gradient_boost,0.798883,test
6,xgboost,0.894663,train
7,xgboost,0.810056,test


12. Compare all the models' performances in a bar chart and conclude. Which model is the best?

Hint: the option `barmode` in plotly's `px.bar()` might be useful 😇

| Méthode               | Type d’ensemble               | Comment fonctionne ?                                                                    | Points forts                                               | Points faibles                                                       |
| --------------------- | ----------------------------- | --------------------------------------------------------------------------------------- | ---------------------------------------------------------- | -------------------------------------------------------------------- |
| **Bagging**           | *Parallel ensemble*           | Plusieurs modèles (arbres) entraînés indépendamment sur des échantillons bootstrap      | Réduit la variance, simple, stable                         | Peut sous-apprendre, ne réduit pas le biais                          |
| **Random Forest**     | Bagging + randomisation       | Arbres baggés + sélection aléatoire de features à chaque split                          | Très robuste, peu sensible à l’overfitting                 | Moins performant que GB/XGB sur données tabulaires complexes         |
| **Gradient Boosting** | *Sequential ensemble*         | Chaque arbre apprend à corriger les erreurs de l’arbre précédent                        | Très performant sur petits datasets, faible biais          | Risque d’overfitting, lent car séquentiel, hyperparamètres sensibles |
| **XGBoost**           | *Optimized Gradient Boosting* | Amélioration du GB : régularisation L1/L2, arbres optimisés, gestion valeurs manquantes | Ultra performant, rapide, régularisé, gère grands datasets | Plus complexe, tuning délicat, peut surapprendre si mal regularisé   |


In [70]:
results_sorted = results.sort_values(by="accuracy", ascending=False)

In [71]:
fig = px.bar(
    results_sorted,
    x="model",
    y="accuracy",
    color="set",
    barmode="group",
    title="Model Performance Comparison (Train vs Test)",
    text="accuracy"
)

fig.update_layout(
    width=900,
    height=600,
    xaxis_title="Model",
    yaxis_title="Accuracy",
    legend_title="Dataset",
)

fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')

fig.show()
